In [ ]:
import datasets
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder


dataset = datasets.load_dataset('Malikeh1375/medical-question-answering-datasets', 'all-processed')

/Users/omaratef/miniconda/envs/custom_llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = dataset['train'].to_pandas()
df['input'] = df['instruction']+ "\n" + df['input']
df = df[['input', 'output']]

df.fillna("" ,inplace = True)


text_data = df['input'].dropna().astype(str).tolist() + df['output'].dropna().astype(str).tolist()

df.head()

,input,output
0,"<bos>If you are a doctor, please answer the me...",<bos>hello and thank you for using chatbot. i ...
1,"<bos>If you are a doctor, please answer the me...",<bos>hello brain tumor may present in various ...
2,<bos>Answer this question truthfully\nWhen to ...,<bos>Call your health care provider if symptom...
3,"<bos>If you are a doctor, please answer the me...","<bos>hi surya, thanks for writing in. melanie ..."
4,"<bos>If you are a doctor, please answer the me...",<bos>hello! i read your question and understan...


In [ ]:
# 1. Initialize an empty BPE tokenizer model
tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=30000, 
    special_tokens=["<unk>", "<pad>", "<bos>", "<eos>"],
    initial_alphabet=ByteLevel.alphabet()
)
# 5. Train the Tokenizer
# You can pass a list of text file paths to train on
tokenizer.train_from_iterator(text_data, trainer)

# 6. Save the trained tokenizer for later use
tokenizer.save("bpe_tokenizer.json")

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

class CustomDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor([
            tokenizer.encode(x).ids for x in X
        ])
        self.Y = torch.tensor([
            tokenizer.encode(y).ids for y in Y
        ])

    def __len__(self):
        return self.X.shape[-1]
    def __getitem__(self, index):
        return self.X[index], self.Y[index]


In [14]:
tokens.ids

[911, 927, 347, 306]

In [16]:
tokenizer.decode(tokens.ids)

'hello how are you'